# Зачетное задание 6.1

Вычисления выполнены без сохранения промежуточных изображений: все основные шаги работают с объектами `cv2` / `numpy`.

In [1]:
from pathlib import Path
from math import floor

import cv2
import numpy as np
import pywt

IMAGE_PATH = Path('input_images/rune-haugseng-UCzjZPCGV1Y-unsplash.jpg')
if not IMAGE_PATH.exists():
    raise FileNotFoundError(f'Не найден файл: {IMAGE_PATH}')

## 1. Чтение, RGB и resize

In [2]:
image_bgr = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)
if image_bgr is None:
    raise ValueError('OpenCV не смог считать изображение')

image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
height, width = image_rgb.shape[:2]

aspect_ratio = width / height
new_width = 2858
new_height = floor(new_width / aspect_ratio)

resized_rgb = cv2.resize(
    image_rgb,
    (new_width, new_height),
    interpolation=cv2.INTER_AREA,
)

print(f'Исходный размер: {width}x{height}')
print(f'Коэффициент width/height: {aspect_ratio}')
print(f'Коэффициент для формы: {aspect_ratio:.3f}')
print(f'Размер после сжатия: {resized_rgb.shape[1]}x{resized_rgb.shape[0]}')

Исходный размер: 5717x4672
Коэффициент width/height: 1.2236729452054795
Коэффициент для формы: 1.224
Размер после сжатия: 2858x2335


## 2. Плотности RGB-гистограмм

Для оценки плотности используется нормированная гистограмма на 256 бинов в диапазоне `0..256`: `np.histogram(..., bins=256, range=(0, 256), density=True)`.

In [3]:
density_results = {}
for idx, channel_name in enumerate('RGB'):
    channel = resized_rgb[:, :, idx]
    density, bin_edges = np.histogram(
        channel.ravel(),
        bins=256,
        range=(0, 256),
        density=True,
    )
    max_idx = int(density.argmax())
    density_results[channel_name] = {
        'max_density': float(density[max_idx]),
        'bin': max_idx,
        'left_edge': float(bin_edges[max_idx]),
        'right_edge': float(bin_edges[max_idx + 1]),
    }

for channel_name, result in density_results.items():
    print(channel_name, result)

max_density = max(result['max_density'] for result in density_results.values())
print(f'Наибольшая плотность: {max_density:.4f}')

R {'max_density': 0.014528960369704934, 'bin': 16, 'left_edge': 16.0, 'right_edge': 17.0}
G {'max_density': 0.02932450029445128, 'bin': 1, 'left_edge': 1.0, 'right_edge': 2.0}
B {'max_density': 0.043916996207347646, 'bin': 2, 'left_edge': 2.0, 'right_edge': 3.0}
Наибольшая плотность: 0.0439


## 3. Линейная нормировка каналов и пиксель

In [4]:
normalized_rgb = np.empty_like(resized_rgb)
for idx, channel_name in enumerate('RGB'):
    channel = resized_rgb[:, :, idx].astype(np.float32)
    normalized_rgb[:, :, idx] = np.round(
        (channel - channel.min()) * 255 / (channel.max() - channel.min())
    ).clip(0, 255).astype(np.uint8)
    print(f'{channel_name}: min={channel.min()}, max={channel.max()}')

row, col = 1196, 1464
pixel_rgb = normalized_rgb[row, col].tolist()
print(f'RGB в normalized_rgb[{row}, {col}]: {pixel_rgb}')

R: min=0.0, max=255.0


G: min=0.0, max=255.0
B: min=0.0, max=255.0
RGB в normalized_rgb[1196, 1464]: [222, 222, 222]


## 4. Два Haar-преобразования и threshold

Так как DWT обычно применяется к двумерному изображению, перед вейвлет-преобразованием нормированное RGB-изображение переводится в grayscale. Затем выполняются два преобразования Хаара, и в `cA1` считаются коэффициенты с модулем меньше `60`.

In [5]:
threshold = 60
normalized_gray = cv2.cvtColor(normalized_rgb, cv2.COLOR_RGB2GRAY)

cA, (cH, cV, cD) = pywt.dwt2(normalized_gray, 'haar')
cA1, (cH1, cV1, cD1) = pywt.dwt2(cA, 'haar')

small_coefficients = int(np.sum(np.abs(cA1) < threshold))
print(f'cA shape: {cA.shape}')
print(f'cA1 shape: {cA1.shape}')
print(f'Количество |cA1| < {threshold}: {small_coefficients}')

cA shape: (1168, 1429)
cA1 shape: (584, 715)
Количество |cA1| < 60: 68864


## 5. Проверка неоднозначных RGB-вариантов

Если применять Haar к каждому RGB-каналу отдельно, получается другое количество. В основной ответ это не идет, но здесь оставлено для контроля.

In [6]:
per_channel_total = 0
for idx, channel_name in enumerate('RGB'):
    cA_channel, _ = pywt.dwt2(normalized_rgb[:, :, idx], 'haar')
    cA1_channel, _ = pywt.dwt2(cA_channel, 'haar')
    count = int(np.sum(np.abs(cA1_channel) < threshold))
    per_channel_total += count
    print(f'{channel_name}: {count}')

print(f'RGB per-channel total: {per_channel_total}')

cA_default, _ = pywt.dwt2(normalized_rgb, 'haar')
cA1_default, _ = pywt.dwt2(cA_default, 'haar')
print(f'pywt.dwt2 default on RGB array: {int(np.sum(np.abs(cA1_default) < threshold))}')

R: 53281


G: 131094
B: 123445
RGB per-channel total: 307820


pywt.dwt2 default on RGB array: 275887


## Ответы для формы

- Коэффициент: `1.224`
- Высота сжатого изображения: `2335`
- Наибольшая плотность: `0.0439`
- R: `222`
- G: `222`
- B: `222`
- Количество коэффициентов: `68864`